# MVTec AD anomaly detection

Train one PatchCore model per product category, then inspect images with an upload panel.
The default configuration uses a pretrained ResNet18 backbone and a 1% coreset.

Run the cells in order. In Colab, select a GPU runtime and edit the Drive paths in section 2.
For local Jupyter, use Python 3.11 or 3.12 and install the repository requirements.

Only normal training images are used for fitting, development, and calibration (60/20/20).
Official test images are reserved for optional evaluation. Training builds a feature memory;
it does not fine-tune the backbone.


## 1 Install packages

In Colab, use a fresh runtime. If installation requests a restart, restart and continue from section 2.
For local Jupyter, install `requirements.txt` in your notebook environment first.


In [ ]:
import sys
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "anomalib==2.3.0", "torch>=2.2", "torchvision>=0.17",
    "numpy>=1.26,<3", "tqdm>=4.66,<5", "ipywidgets>=8,<9", "pandas>=2,<3",
    "scikit-learn>=1.4,<2", "matplotlib>=3.8,<4", "pillow>=10,<12",
])


## 2 Set project and dataset paths

In Colab, mount Drive and change `PROJECT_DIR` to your project folder.
Locally, the default is the notebook's working directory.

Set `DATA_SOURCE` to the extracted dataset folder or its ZIP/TAR archive.
Models are saved under `RUN_DIR`; `WORKSPACE` holds temporary data.


In [ ]:
from pathlib import Path
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/mvtec-all-categories")  # Edit this path.
    WORKSPACE = Path("/content/mvtec_work")
else:
    PROJECT_DIR = Path.cwd()  # Change this if your dataset is elsewhere.
    WORKSPACE = PROJECT_DIR / "mvtec_work"

DATA_SOURCE = PROJECT_DIR / "mvtec_anomaly_detection"
RUN_DIR = PROJECT_DIR / "runs" / "patchcore_resnet18_256_r001_seed42"
SOURCE_DIR = Path("/content") if IN_COLAB else Path.cwd()

assert DATA_SOURCE.exists(), f"Dataset path does not exist: {DATA_SOURCE}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Dataset source:", DATA_SOURCE)
print("Saved results:", RUN_DIR)


## 3 Write the Python modules

These cells write the training and inference modules to the working directory.
Rerun them in a fresh Colab runtime.


In [ ]:
%%writefile {SOURCE_DIR}/data_utils.py
"""Dataset preparation and calibration helpers; no model dependencies."""
from pathlib import Path
import hashlib
import json
import os
import shutil
import stat
import tarfile
import zipfile
import numpy as np

CATEGORIES = (
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
)
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def image_paths(folder):
    return sorted(p for p in Path(folder).iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def find_dataset_root(folder):
    """Find a unique category parent up to four folder levels below folder."""
    folder = Path(folder).expanduser().resolve()
    if not folder.is_dir():
        raise FileNotFoundError(f"Not a folder: {folder}")
    candidates = []
    for current, dirs, _ in os.walk(folder):
        current = Path(current)
        depth = len(current.relative_to(folder).parts)
        if depth >= 4:
            dirs[:] = []
        if any((current / cat / "train" / "good").is_dir() for cat in CATEGORIES):
            candidates.append(current)
            dirs[:] = []
    if len(candidates) != 1:
        raise ValueError(
            f"Expected one dataset root, found {len(candidates)}: {candidates}. "
            "Set DATA_SOURCE to the exact extracted dataset folder. "
            "If it is still compressed, set DATA_SOURCE to the archive file."
        )
    return candidates[0]

def validate_categories(root, categories=CATEGORIES):
    root = Path(root)
    missing = [cat for cat in categories if not (root / cat / "train" / "good").is_dir()]
    if missing:
        raise ValueError(f"Missing categories: {missing}. Expected category/train/good.")
    counts = {cat: len(image_paths(root / cat / "train" / "good")) for cat in categories}
    if any(n < 10 for n in counts.values()):
        raise ValueError(f"Too few normal images for fitting and calibration: {counts}")
    return counts

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

def safe_extract(archive_path, destination):
    """Extract ZIP or TAR into destination, rejecting escaped paths and ZIP links."""
    archive_path, destination = Path(archive_path), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(destination):
                    raise ValueError(f"Unsafe archive path: {item.filename}")
                if stat.S_ISLNK(item.external_attr >> 16):
                    raise ValueError(f"ZIP symbolic link not supported: {item.filename}")
            archive.extractall(destination)
    elif tarfile.is_tarfile(archive_path):
        if not hasattr(tarfile, "data_filter"):
            raise RuntimeError("Use Python 3.12+ for filtered TAR extraction.")
        with tarfile.open(archive_path, "r:*") as archive:
            archive.extractall(destination, filter="data")
    else:
        raise ValueError("Provide an extracted folder or a ZIP/TAR archive.")

def prepare_source(source, workspace):
    """Use an extracted Drive folder or copy/extract one specified archive locally."""
    source, workspace = Path(source), Path(workspace)
    if source.is_dir():
        return find_dataset_root(source)
    if not source.is_file():
        raise FileNotFoundError(f"DATA_SOURCE does not exist: {source}")
    signature = {"source": str(source.resolve()), "bytes": source.stat().st_size,
                 "mtime_ns": source.stat().st_mtime_ns}
    destination = workspace / "archive_data"
    marker = workspace / "archive_ready.json"
    if marker.exists() and json.loads(marker.read_text()) == signature:
        return find_dataset_root(destination)
    if destination.exists():
        shutil.rmtree(destination)
    if marker.exists():
        marker.unlink()
    workspace.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(workspace).free < source.stat().st_size * 3:
        raise OSError("Insufficient runtime disk space for archive copy and extraction.")
    local_archive = workspace / source.name
    shutil.copy2(source, local_archive)
    print("Extracting archive into runtime storage; this can take several minutes.")
    safe_extract(local_archive, destination)
    root = find_dataset_root(destination)
    atomic_json(marker, signature)
    local_archive.unlink()
    return root

def stage_category(source_root, category, workspace):
    """Copy only one category at a time; skip unrelated notebooks and output files."""
    source = Path(source_root) / category
    local_parent = Path(workspace) / "active_category"
    local_parent.mkdir(parents=True, exist_ok=True)
    destination = local_parent / category
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir()
    for role in ("train", "test", "ground_truth"):
        if (source / role).is_dir():
            shutil.copytree(source / role, destination / role)
    return destination

def make_normal_manifest(category_root, seed=42):
    """Disjoint 60/20/20 groups from official train/good; hash contents for resume."""
    category_root = Path(category_root)
    paths = image_paths(category_root / "train" / "good")
    if len(paths) < 10:
        raise ValueError("Need at least ten normal training images.")
    order = np.random.default_rng(seed).permutation(len(paths))
    n_fit, n_dev = round(.60 * len(paths)), round(.20 * len(paths))
    rows = []
    for rank, index in enumerate(order):
        path = paths[index]
        split = "fit" if rank < n_fit else ("dev" if rank < n_fit + n_dev else "calibration")
        rows.append({"path": str(path.relative_to(category_root)), "split": split,
                     "sha256": sha256_file(path)})
    fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()
    return rows, fingerprint

def calibration_record(scores, alpha=.05):
    scores = np.asarray(scores, dtype=float)
    if scores.ndim != 1 or len(scores) < 2 or not np.isfinite(scores).all():
        raise ValueError("Calibration needs at least two finite image scores.")
    if not 0 < alpha < 1:
        raise ValueError("alpha must be between zero and one.")
    threshold = float(np.quantile(scores, 1 - alpha, method="higher"))
    return {"threshold": threshold, "target_fpr": float(alpha), "comparison": ">",
            "count": len(scores), "observed_calibration_fpr": float((scores > threshold).mean()),
            "map_display_max": None}



In [ ]:
%%writefile {SOURCE_DIR}/mvtec_patchcore.py
"""Anomalib 2.3.0 PatchCore with explicit normal-only splits and raw scoring."""
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import os
import random
import threading
import time
from datetime import datetime, timezone
import numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
from anomalib.models.image.patchcore.torch_model import PatchcoreModel
from data_utils import (CATEGORIES, image_paths, atomic_json, make_normal_manifest,
                        calibration_record, stage_category, validate_categories)

EXPECTED_ANOMALIB = "2.3.0"
FORMAT_VERSION = 1
DEFAULT_CONFIG = {"backbone": "resnet18", "layers": ["layer2", "layer3"],
                  "image_size": 256, "coreset_ratio": .01, "num_neighbors": 9,
                  "batch_size": 4, "seed": 42, "target_fpr": .05,
                  "preprocess": "RGB_full_resize_bilinear_ImageNet_normalization_v1"}

def check_version():
    version = importlib.metadata.version("anomalib")
    if version != EXPECTED_ANOMALIB:
        raise RuntimeError(f"Use anomalib=={EXPECTED_ANOMALIB}; installed: {version}")

def environment():
    return {"torch": str(torch.__version__), "anomalib": EXPECTED_ANOMALIB,
            "numpy": str(np.__version__), "device": (
                torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
            "saved_utc": datetime.now(timezone.utc).isoformat()}

def make_transform(config):
    return transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"]),
                          interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
        transforms.Normalize(mean=[.485, .456, .406], std=[.229, .224, .225]),
    ])

def decode_image(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB").copy()

class ImageDataset(Dataset):
    def __init__(self, paths, config):
        self.paths = [Path(p) for p in paths]
        self.transform = make_transform(config)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        return self.transform(decode_image(self.paths[index]))

def loader(paths, config, batch_size=None):
    return DataLoader(ImageDataset(paths, config), shuffle=False, num_workers=0,
                      batch_size=batch_size or config["batch_size"],
                      pin_memory=torch.cuda.is_available())

def new_model(config, pretrained=True):
    check_version()
    return PatchcoreModel(layers=config["layers"], backbone=config["backbone"],
                         pre_trained=pretrained, num_neighbors=config["num_neighbors"])

@torch.no_grad()
def score_paths(model, paths, config, device, desc="Scoring"):
    scores, map_peaks = [], []
    model.eval()
    for images in tqdm(loader(paths, config, batch_size=1), desc=desc):
        prediction = model(images.to(device))
        scores.extend(prediction.pred_score.detach().cpu().reshape(-1).tolist())
        peaks = prediction.anomaly_map.flatten(1).amax(1)
        map_peaks.extend(peaks.detach().cpu().tolist())
    values = np.asarray(scores, dtype=float)
    if not np.isfinite(values).all():
        raise RuntimeError("Non-finite model scores; check data and memory bank.")
    return values, np.asarray(map_peaks, dtype=float)

def write_checkpoint(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    torch.save(payload, temporary)
    os.replace(temporary, path)

def read_checkpoint(path):
    payload = torch.load(Path(path), map_location="cpu", weights_only=True)
    if payload.get("format_version") != FORMAT_VERSION:
        raise RuntimeError("Unsupported checkpoint format.")
    if payload.get("anomalib_version") != EXPECTED_ANOMALIB:
        raise RuntimeError("Checkpoint/library version mismatch.")
    bank = payload["state_dict"]["memory_bank"]
    if bank.ndim != 2 or bank.shape[0] < 1 or not torch.isfinite(bank).all():
        raise RuntimeError("Checkpoint contains an invalid memory bank.")
    if not np.isfinite(payload["calibration"]["threshold"]):
        raise RuntimeError("Checkpoint threshold is invalid.")
    return payload

def load_model(path, device):
    payload = read_checkpoint(path)
    if Path(path).stem != payload["category"]:
        raise RuntimeError("Checkpoint filename and saved category do not match.")
    model = new_model(payload["config"], pretrained=False)
    # Make buffer loading explicit; do not depend on automatic buffer resizing.
    model.memory_bank = torch.empty_like(payload["state_dict"]["memory_bank"])
    model.load_state_dict(payload["state_dict"], strict=True)
    model.to(device).eval()
    return model, payload

def train_category(local_category, run_dir, category, config, force=False):
    check_version()
    config = dict(config)
    if config["backbone"] not in {"resnet18", "wide_resnet50_2"}:
        raise ValueError("This preprocessing is supported for the configured ImageNet backbones only.")
    if not 0 < config["coreset_ratio"] <= 1:
        raise ValueError("coreset_ratio must be in (0, 1].")
    local_category, run_dir = Path(local_category), Path(run_dir)
    rows, fingerprint = make_normal_manifest(local_category, config["seed"])
    manifest_path = run_dir / "manifests" / f"{category}.json"
    checkpoint = run_dir / "models" / f"{category}.pt"
    if checkpoint.exists() and not force:
        existing = read_checkpoint(checkpoint)
        if existing["config"] != config or existing["data_fingerprint"] != fingerprint:
            raise RuntimeError(f"{category}: saved configuration/data differ. Use a new RUN_DIR.")
        atomic_json(manifest_path, rows)
        print(f"{category}: verified saved model; skipping fitting.")
        return checkpoint
    atomic_json(manifest_path, rows)
    groups = {name: [local_category / r["path"] for r in rows if r["split"] == name]
              for name in ("fit", "dev", "calibration")}
    assert not set(groups["fit"]) & set(groups["calibration"])
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = new_model(config).to(device)
    # The outer model must be in training mode to collect patch embeddings.
    # Freeze backbone batch normalization and all other pretrained behaviour.
    model.train()
    model.feature_extractor.eval()
    start = time.perf_counter()
    try:
        with torch.no_grad():
            for images in tqdm(loader(groups["fit"], config), desc=f"{category}: normal features"):
                model(images.to(device))
            print(f"{category}: selecting representative memory features...")
            model.subsample_embedding(sampling_ratio=config["coreset_ratio"])
        model.eval()
        calibration_scores, map_peaks = score_paths(
            model, groups["calibration"], config, device, f"{category}: calibration")
        cal = calibration_record(calibration_scores, config["target_fpr"])
        cal["map_display_max"] = max(float(np.quantile(map_peaks, .99)), 1e-8)
        cal["scores"] = calibration_scores.tolist()
        payload = {
            "format_version": FORMAT_VERSION, "category": category, "config": config,
            "anomalib_version": EXPECTED_ANOMALIB, "data_fingerprint": fingerprint,
            "calibration": cal, "split_counts": {k: len(v) for k, v in groups.items()},
            "memory_vectors": int(model.memory_bank.shape[0]),
            "fit_and_calibration_seconds": float(time.perf_counter() - start),
            "environment": environment(),
            "state_dict": {k: value.detach().cpu() for k, value in model.state_dict().items()},
        }
        write_checkpoint(checkpoint, payload)
        summary = {k: v for k, v in payload.items() if k != "state_dict"}
        atomic_json(run_dir / "calibration" / f"{category}.json", summary)
        print(f"Saved {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MiB)")
        return checkpoint
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def train_all(source_root, run_dir, workspace, config=None, categories=CATEGORIES, force=False):
    config = dict(DEFAULT_CONFIG if config is None else config)
    validate_categories(source_root, categories)
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    config_path = run_dir / "config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise RuntimeError("RUN_DIR has another configuration. Choose a new RUN_DIR.")
    atomic_json(config_path, config)
    completed = []
    for category in categories:
        local_category = stage_category(source_root, category, workspace)
        try:
            train_category(local_category, run_dir, category, config, force)
            completed.append(category)
            atomic_json(run_dir / "progress.json", {"completed_in_this_run": completed})
        finally:
            import shutil
            shutil.rmtree(local_category, ignore_errors=True)
    return run_dir

def evaluate_all(source_root, run_dir, workspace, categories=CATEGORIES):
    """Image-level evaluation only; not used to select thresholds or tune settings."""
    from sklearn.metrics import roc_auc_score, confusion_matrix
    import pandas as pd
    run_dir = Path(run_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    summaries = []
    for category in categories:
        local = stage_category(source_root, category, workspace)
        model = None
        try:
            model, payload = load_model(run_dir / "models" / f"{category}.pt", device)
            _, fingerprint = make_normal_manifest(local, payload["config"]["seed"])
            if fingerprint != payload["data_fingerprint"]:
                raise RuntimeError("Evaluation source does not match fitted category data.")
            test_dir = local / "test"
            if not test_dir.is_dir():
                raise FileNotFoundError(f"Missing official test directory: {test_dir}")
            records, paths = [], []
            for group in sorted(p for p in test_dir.iterdir() if p.is_dir()):
                for path in image_paths(group):
                    paths.append(path)
                    records.append({"path": str(path.relative_to(local)),
                                    "defect_group": group.name, "label": int(group.name != "good")})
            if not paths:
                raise ValueError(f"No test images for {category}")
            scores, _ = score_paths(model, paths, payload["config"], device, f"{category}: test")
            labels = np.array([r["label"] for r in records])
            predicted = (scores > payload["calibration"]["threshold"]).astype(int)
            tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
            metric = {"category": category,
                      "image_auroc": float(roc_auc_score(labels, scores)) if len(set(labels)) == 2 else None,
                      "recall": float(tp / (tp + fn)) if tp + fn else None,
                      "false_positive_rate": float(fp / (fp + tn)) if fp + tn else None,
                      "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                      "threshold": payload["calibration"]["threshold"]}
            for row, score, decision in zip(records, scores, predicted):
                row.update(score=float(score), decision=int(decision))
            atomic_json(run_dir / "evaluation" / f"{category}_predictions.json", records)
            atomic_json(run_dir / "evaluation" / f"{category}_metrics.json", metric)
            summaries.append(metric)
        finally:
            del model
            import shutil
            shutil.rmtree(local, ignore_errors=True)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    table = pd.DataFrame(summaries)
    table.to_csv(run_dir / "evaluation" / "summary.csv", index=False)
    return table

class Inspector:
    """Keep only the current category on the device, with serialized inference."""
    def __init__(self, run_dir, device=None):
        self.run_dir = Path(run_dir)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = self.payload = self.category = None
        self.lock = threading.Lock()
    def categories(self):
        return [cat for cat in CATEGORIES if (self.run_dir / "models" / f"{cat}.pt").exists()]
    @torch.no_grad()
    def inspect(self, image, category):
        if not isinstance(image, Image.Image):
            raise ValueError("Upload an image first.")
        if image.width * image.height > 25_000_000:
            raise ValueError("Use an image smaller than 25 megapixels.")
        if category not in self.categories():
            raise ValueError("The selected category has no saved model.")
        with self.lock:
            if self.category != category:
                self.model = None
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
                self.model, self.payload = load_model(
                    self.run_dir / "models" / f"{category}.pt", self.device)
                self.category = category
            image = ImageOps.exif_transpose(image).convert("RGB")
            tensor = make_transform(self.payload["config"])(image).unsqueeze(0).to(self.device)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            prediction = self.model(tensor)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = (time.perf_counter() - start) * 1000
            score = float(prediction.pred_score.detach().cpu().reshape(-1)[0])
            raw_map = prediction.anomaly_map.detach().cpu().numpy().reshape(
                self.payload["config"]["image_size"], self.payload["config"]["image_size"])
            if not np.isfinite(score) or not np.isfinite(raw_map).all():
                raise RuntimeError("Invalid model output.")
            # Fixed category display scale comes exclusively from calibration normals.
            scale = max(self.payload["calibration"]["map_display_max"] * 2, 1e-8)
            display_map = np.clip(raw_map / scale, 0, 1)
            from matplotlib import colormaps
            heatmap = Image.fromarray((colormaps["inferno"](display_map)[..., :3] * 255).astype(np.uint8))
            base = image.resize(heatmap.size, Image.Resampling.BILINEAR)
            overlay = Image.blend(base, heatmap, .45)
            threshold = self.payload["calibration"]["threshold"]
            result = {"category": category, "decision": (
                "Review recommended" if score > threshold else "No anomaly detected at this threshold"),
                "anomaly_score": score, "threshold": threshold,
                "scoring_ms_excluding_load_and_preprocessing": round(elapsed, 2),
                "model": "PatchCore / " + self.payload["config"]["backbone"]}
            return base, heatmap, overlay, result


## 4 Configure training

The defaults use ResNet18 and a 1% coreset. Change `RUN_DIR` when changing model settings.
Training processes one category at a time.


In [ ]:
import sys
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
import torch
import numpy as np
from data_utils import CATEGORIES, prepare_source, validate_categories
from mvtec_patchcore import (DEFAULT_CONFIG, check_version, train_all,
                            evaluate_all, Inspector, environment)

check_version()
CONFIG = dict(DEFAULT_CONFIG)
# CONFIG["batch_size"] = 1  # Use this if batches cause GPU out-of-memory errors.
SELECTED_CATEGORIES = list(CATEGORIES)  # All 15 by default.
FORCE_RETRAIN = False
print(environment())
print("Configuration:", CONFIG)
if not torch.cuda.is_available():
    print("WARNING: CPU selected. Data checks work; all-category fitting may be slow.")


## 5 Validate the dataset

Check the selected categories and their normal training image counts.
Archives are extracted into temporary storage; extracted datasets are staged one category at a time.


In [ ]:
SOURCE_ROOT = prepare_source(DATA_SOURCE, WORKSPACE)
counts = validate_categories(SOURCE_ROOT, SELECTED_CATEGORIES)
print("Dataset root:", SOURCE_ROOT)
for category, count in counts.items():
    print(f"{category:12s} {count:4d} normal training images")
print("Category count:", len(counts))
assert len(counts) == len(SELECTED_CATEGORIES)


## 6 Train and reload the first category

Check training, calibration, checkpoint saving, and inference before running every category.


In [ ]:
from PIL import Image
from data_utils import image_paths

first_category = SELECTED_CATEGORIES[0]
train_all(SOURCE_ROOT, RUN_DIR, WORKSPACE, CONFIG,
          categories=[first_category], force=FORCE_RETRAIN)
example_path = image_paths(SOURCE_ROOT / first_category / "train" / "good")[0]
with Image.open(example_path) as image:
    inspector_check = Inspector(RUN_DIR)
    original, heatmap, overlay, result = inspector_check.inspect(image, first_category)
assert original.size == (CONFIG["image_size"], CONFIG["image_size"])
assert np.isfinite(result["anomaly_score"])
print("Checkpoint reload check passed:", result)
del inspector_check
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 7 Train the selected categories

Each completed category is saved immediately. After reconnecting, rerun sections 1–5 and this cell.
Matching checkpoints are reused; interrupted categories start again. Use a new run folder for different settings.


In [ ]:
train_all(SOURCE_ROOT, RUN_DIR, WORKSPACE, CONFIG,
          categories=SELECTED_CATEGORIES, force=FORCE_RETRAIN)
saved_categories = Inspector(RUN_DIR).categories()
print("Saved category models:", saved_categories)
assert set(saved_categories) >= set(SELECTED_CATEGORIES)
print("Training and calibration completed. Saved models:", RUN_DIR / "models")


## 8 Inspect an image

Choose the product category, upload a photo, and click **Inspect image**.
Heatmaps use a fixed scale based on that category's calibration images.
Scores are not defect probabilities. Lighting and background changes can affect results.


In [ ]:
import io
import json
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

inspector = Inspector(RUN_DIR)
available = inspector.categories()
assert available, "Run the training cells first."
category_control = widgets.Dropdown(options=available, description="Category:")
upload_control = widgets.FileUpload(accept="image/*", multiple=False)
inspect_button = widgets.Button(description="Inspect image", button_style="primary")
result_area = widgets.Output()

def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()

def inspect_upload(_):
    inspect_button.disabled = True
    try:
        with result_area:
            clear_output(wait=True)
            if not upload_control.value:
                print("Upload an image first.")
                return
            items = upload_control.value
            item = next(iter(items.values())) if isinstance(items, dict) else items[0]
            content = bytes(item["content"])
            if len(content) > 20 * 1024 * 1024:
                raise ValueError("Upload a file smaller than 20 MiB.")
            with Image.open(io.BytesIO(content)) as photo:
                if photo.width * photo.height > 25_000_000:
                    raise ValueError("Use an image smaller than 25 megapixels.")
                original, heatmap, overlay, result = inspector.inspect(photo, category_control.value)
            display(widgets.HBox([
                widgets.VBox([widgets.Label(label), widgets.Image(value=png_bytes(img), format="png")])
                for label, img in [("Resized input", original), ("Heatmap", heatmap), ("Overlay", overlay)]
            ]))
            print(result["decision"])
            print(json.dumps(result, indent=2))
    except Exception as error:
        with result_area:
            print("Inspection failed:", error)
    finally:
        inspect_button.disabled = False

inspect_button.on_click(inspect_upload)
display(widgets.HTML("<h3>Visual Product Inspection</h3>"),
        category_control, upload_control, inspect_button, result_area)


## 9 Evaluate on the official test set

Run after choosing your model settings. This saves image AUROC, recall, false-positive rate,
error counts, and individual scores. Pixel-level metrics are not included.
The calibration target does not guarantee the same false-positive rate on new images.


In [ ]:
# Release the current UI model if the upload panel was opened.
import gc
if "inspector" in globals():
    inspector.model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

summary = evaluate_all(SOURCE_ROOT, RUN_DIR, WORKSPACE,
                       categories=SELECTED_CATEGORIES)
display(summary)
print("Category-macro averages:")
print(summary[["image_auroc", "recall", "false_positive_rate"]].mean())
print("Results:", RUN_DIR / "evaluation")
# Recreate the Inspector; rerun the UI cell if you want to inspect more images.
inspector = Inspector(RUN_DIR)


## 10 Export the standalone app

Save the Python modules and requirements alongside the trained models.
Download the run folder to use the Gradio app on another computer.


In [ ]:
%%writefile {SOURCE_DIR}/app.py
"""Gradio interface for saved MVTec PatchCore models."""
import argparse
from pathlib import Path
import gradio as gr
from mvtec_patchcore import Inspector

def build_app(run_dir):
    inspector = Inspector(run_dir)
    categories = inspector.categories()
    if not categories:
        raise RuntimeError(
            f"No trained category models found in {Path(run_dir).resolve() / 'models'}. "
            "Train models with the notebook or download a trained run folder. "
            "Pass its path using --run-dir. See README.md."
        )
    def inspect(image, category):
        try:
            return inspector.inspect(image, category)
        except Exception as error:
            raise gr.Error(str(error)) from error
    with gr.Blocks(title="Visual Product Inspection") as demo:
        gr.Markdown("# Visual Product Inspection\nSelect the product category, upload a photo, and inspect the highlighted regions.")
        with gr.Row():
            category = gr.Dropdown(categories, value=categories[0], label="Product category")
            image = gr.Image(type="pil", label="Upload product image", sources=["upload"])
        button = gr.Button("Inspect image", variant="primary")
        with gr.Row():
            original = gr.Image(label="Resized input")
            heatmap = gr.Image(label="Anomaly heatmap")
            overlay = gr.Image(label="Highlighted regions")
        result = gr.JSON(label="Inspection result")
        gr.Markdown("Scores are not defect probabilities. Use images similar to the training data.")
        button.click(inspect, [image, category], [original, heatmap, overlay, result], concurrency_limit=1)
    return demo

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--run-dir", required=True, help="Folder containing models/*.pt")
    parser.add_argument("--port", type=int, default=7860)
    args = parser.parse_args()
    build_app(args.run_dir).queue().launch(server_name="127.0.0.1", server_port=args.port, share=False)


In [ ]:
import shutil
bundle_source = RUN_DIR / "source"
bundle_source.mkdir(exist_ok=True)
for filename in ["data_utils.py", "mvtec_patchcore.py", "app.py"]:
    shutil.copy2(SOURCE_DIR / filename, bundle_source / filename)
requirements = 'anomalib==2.3.0\ntorch>=2.2\ntorchvision>=0.17\ngradio==5.49.1\nnumpy>=1.26,<3\npillow>=10,<12\npandas>=2,<3\nscikit-learn>=1.4,<2\nmatplotlib>=3.8,<4\ntqdm>=4.66,<5\nipywidgets>=8,<9\n'
(bundle_source / "requirements.txt").write_text(requirements)
print("Saved standalone app source:", bundle_source)
print("Download the whole run folder, including models/ and source/.")


### Run the exported app

Use Python 3.11 or 3.12. In the downloaded run folder's `source` directory:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python app.py --run-dir ..
```

On Windows, create the environment with `py -3.12 -m venv .venv` and activate it with `.venv\Scripts\activate`.
Open http://127.0.0.1:7860. Saved models are required; the original dataset is not.

### References

- [MVTec AD dataset and licence](https://www.mvtec.com/research-teaching/datasets/mvtec-ad)
- [PatchCore paper](https://arxiv.org/abs/2106.08265)
- [Anomalib 2.3.0](https://pypi.org/project/anomalib/2.3.0/)

Full training and the live UI still need verification. Record your environment versions and evaluation results.
